모델 배포 개론 02  
  Last modified : 2026.08  
  작성 : 박광석 (모두의연구소)  
  수정 : 김지성, 박기웅 (모두의연구소)  
  수정 : 김민욱 (수강생, 2026.08.13) — 아래 "내가 고친 곳" 참조

### 내가 고친 곳 (2026.08.13, 김민욱)

원본은 설명에 적힌 값과 실제로 보내는 값이 달랐다.
`schemas.py` 의 `pixel_values` 설명은 "0.0~1.0 범위" 라고 되어 있는데,
5.6절 테스트가 실제로 보내는 값은 정규화된 `-0.4242 ~ 2.8215` 였다.
`Field` 는 개수만 검사하고 값의 범위는 안 보기 때문에,
설명을 믿고 0~1 을 보내도 422 가 안 나고 그냥 200 이 나간다.

얼마나 차이가 나는지 테스트셋 10,000장으로 재봤다.

| | 정규화된 값을 보냄 | 설명을 믿고 0~1 을 보냄 |
|---|---|---|
| 정확도 | 99.06% | 98.80% (26장 손해) |
| 평균 확신도 | 0.9913 | 0.7920 |
| 답이 갈린 이미지 | | 45장 / 10,000 |

정확도 손해는 생각보다 작았다. 대신 확신도가 많이 떨어진다.
확신도가 얼마 이하면 사람이 검토하게 하는 식으로 기준을 걸어둔 서비스라면
여기서 문제가 될 것 같다. 그런데 200 이 나가고 스키마도 통과하고
답도 대부분 맞아서 자동 테스트로는 안 걸린다.

원인을 찾아보니 `model_utils.py` 에 `preprocess` 를 만들어놓고
`main.py` 가 그걸 import 조차 안 하고 있었다.
전처리를 서버가 아니라 클라이언트가 해야 하는 상태였다.

그래서 이렇게 고쳤다.

1. `model_utils.py` — `to_model_input()` 추가. 정규화를 서버가 하도록 했다.
2. `schemas.py` — 원소마다 `ge=0.0, le=1.0` 을 걸어 범위를 검사하게 했다.
3. `main.py` — 직접 텐서를 만들던 부분을 `to_model_input()` 호출로 바꿨다.
4. 5.6절 테스트 셀 — `Normalize` 를 뺐다. 이제 0~1 값을 그대로 보낸다.

고친 뒤 다시 재보니 정확도 99.06% / 평균 확신도 0.9913 으로
원래대로 돌아왔고, 범위를 벗어난 값은 422 로 막힌다.

# 코랩버전 사전 실행 모델 저장 day1    
    
로컬이면 할 필요 없습니다.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
# ===== 모델 정의 (섹션 4와 동일) =====
class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
# ===== 하이퍼파라미터 =====
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 3            # 실습용이므로 3 에포크만 학습합니다

# ===== 디바이스 설정 =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

In [ ]:
# ===== 데이터 준비 =====
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))   # MNIST 평균/표준편차
])

train_dataset = datasets.MNIST(
    root="data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root="data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"학습 데이터: {len(train_dataset):,}장")
print(f"테스트 데이터: {len(test_dataset):,}장")

In [ ]:
model = SimpleClassifier(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # 200 배치마다 진행 상황 출력
        if (batch_idx + 1) % 200 == 0:
            print(f"  Epoch {epoch} [{batch_idx+1}/{len(train_loader)}] "
                  f"Loss: {running_loss/(batch_idx+1):.4f} "
                  f"Acc: {100.*correct/total:.1f}%")

    # 에포크 종료 시 요약
    train_acc = 100. * correct / total
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch}/{EPOCHS} 완료 — Loss: {avg_loss:.4f}, Acc: {train_acc:.1f}%\n")

In [ ]:
import os
os.makedirs("models", exist_ok=True)

# 모델을 CPU로 이동 (배포 환경에서는 GPU가 없을 수 있으므로)
model_cpu = model.cpu()
model_cpu.eval()

# 추론 비교용 테스트 입력
test_input = test_dataset[0][0].unsqueeze(0)   # 첫 번째 테스트 이미지
test_label = test_dataset[0][1]                 # 정답 레이블

print(f"테스트 입력 크기: {test_input.shape}")
print(f"정답 레이블: {test_label}")

In [ ]:
# 저장 전 원본 모델의 추론 결과를 기록해 둡니다
with torch.no_grad():
    original_output = model_cpu(test_input)
    original_pred = original_output.argmax(dim=1).item()
    original_conf = torch.softmax(original_output, dim=1).max().item()

print(f"원본 모델 예측: {original_pred} (확신도: {original_conf:.4f})")
print(f"정답:          {test_label}")
print(f"정답 여부:      {'✅ 맞음' if original_pred == test_label else '❌ 틀림'}")


In [ ]:
# state_dict 저장
torch.save(model_cpu.state_dict(), "models/mnist_state_dict.pth")
print(f"✅ state_dict 저장 완료: {os.path.getsize('models/mnist_state_dict.pth') / 1024:.1f} KB")

# Day 2 — FastAPI 기초와 데이터 처리


## 1. 들어가며: 모델 배포에 FastAPI를 선택하는 이유

> **학습 목표**
> - FastAPI가 무엇이고, 어떤 특징을 가지는지 설명할 수 있습니다.
> - FastAPI와 다른 프레임워크의 차이를 확인하고, 모델 배포에 FastAPI가 적합한 이유를 알 수 있습니다.
> - FastAPI 서버를 최소한의 코드로 실행할 수 있습니다.
> - uvicorn의 역할을 이해합니다.

### 1.1 지난 흐름 리마인드

앞서 다음을 완료했습니다:

```
✅ 가상환경과 requirements.txt로 환경을 통제했습니다.
✅ RESTful API의 개념(URL, HTTP 메서드, 상태 코드, JSON)을 이해했습니다.
✅ PyTorch 모델을 직렬화하여 파일로 저장했습니다.
✅ 추론 함수를 app/model_utils.py로 분리해 두었습니다.
```

오늘 할 일은 명확합니다:

> **app/model_utils.py의 `predict()` 함수를 HTTP 요청으로 호출할 수 있도록 만드는 것.**

이를 위해 API 서버 프레임워크가 필요합니다. 이 과정에서는 **FastAPI**를 사용합니다.


### 1.2 FastAPI란 무엇입니까?

FastAPI는 Python으로 API 서버를 만들기 위한 **현대적인 웹 프레임워크**입니다.
2018년에 Sebastián Ramírez가 만들었으며, 이후 빠르게 성장하여
현재 Python 웹 프레임워크 중 가장 높은 성장세를 보이고 있습니다.

이름에 "Fast"가 들어간 이유는 두 가지입니다:

```
1. Fast to run   — 실행 속도가 빠릅니다 (비동기 기반, Starlette + Uvicorn)
2. Fast to code  — 개발 속도가 빠릅니다 (타입 힌트 기반 자동화)
```

### 1.3 모델 배포에 FastAPI가 적합한 이유

Python API 프레임워크는 Flask, Django, FastAPI 등 여러 가지가 있습니다.
이 과정에서 FastAPI를 선택한 이유는, **모델 배포에서 자주 필요한 기능들이 기본 내장**되어 있기 때문입니다.

FastAPI의 세 가지 핵심 강점을 먼저 살펴보고,
이후 다른 프레임워크와의 비교는 참고 수준으로 정리하겠습니다.

#### 강점 1: 자동 데이터 검증 (Pydantic)

모델에 잘못된 입력이 들어오면 에러가 발생합니다.
FastAPI는 **Pydantic**이라는 라이브러리를 내장하고 있어, 요청이 모델에 도달하기 전에 입력을 자동으로 검증합니다.

> 아래 코드를 직접 실행해 보세요.  
> Pydantic은 검증 실패 시 `ValidationError`를 발생시키며,  
> **어떤 필드가, 왜 잘못되었는지** 상세하게 알려줍니다.  
> FastAPI에서는 이 검증이 요청 수신 시 자동으로 수행되어 422 에러로 반환됩니다.

In [ ]:
# 입력 스키마를 클래스로 선언합니다
from pydantic import BaseModel, Field, ValidationError

class PredictRequest(BaseModel):
    text: str = Field(..., min_length=1)    # 빈 문자열 불가

# FastAPI가 자동으로 처리하는 것들:
# - text 필드가 없으면 → 422 에러 + "field required" 메시지
# - text가 문자열이 아니면 → 422 에러 + "string type expected" 메시지
# - text가 빈 문자열이면 → 422 에러 + "min_length" 메시지

In [ ]:
# 정상 입력 — 통과
req = PredictRequest(text="이 영화 재밌다")
print(f"✅ 정상: {req.text}")

In [ ]:
# 에러 1: text 필드 누락
try:
    PredictRequest()
except ValidationError as e:
    print(f"\n❌ 필드 누락:\n{e}")
# Field required

In [ ]:
# 에러 2: 잘못된 타입 (문자열이어야 하는데 리스트를 전달)
try:
    PredictRequest(text=["이것은", "리스트"])
except ValidationError as e:
    print(f"\n❌ 타입 오류:\n{e}")
# Input should be a valid string

In [ ]:
# 에러 3: 빈 문자열 (min_length=1 위반)
try:
    PredictRequest(text="")
except ValidationError as e:
    print(f"\n❌ 빈 문자열:\n{e}")
# String should have at least 1 character


if/else로 검증 로직을 일일이 작성할 필요가 없습니다.  
타입과 조건을 **선언**하기만 하면, 검증은 FastAPI가 담당합니다.  

#### 강점 2: 자동 API 문서화 (Swagger UI)


코드를 작성하면 API 문서가 자동으로 생성됩니다.  
브라우저에서 `/docs` 경로에 접속하면 Swagger UI가 나타나며,  
별도의 도구 없이 **브라우저에서 바로 API를 테스트**할 수 있습니다.

![image.png](images/nb/nb_01_792c32fd.png)

프론트엔드 개발자나 동료에게 별도 문서를 작성할 필요가 없습니다.  

#### 강점 3: 비동기 처리 (async/await)

모델 추론은 시간이 걸리는 작업입니다.  
동기 방식의 서버는 추론이 끝날 때까지 다른 요청을 받지 못합니다.  
FastAPI는 **비동기(async/await) 기반**이므로, 동시에 여러 요청을 처리할 수 있습니다.

![image.png](images/nb/nb_02_62db7a07.png)

#### 참고: 다른 프레임워크와의 비교


![image.png](images/nb/nb_03_7cc9f5de.png)

* Django는 3.1부터 비동기를 지원하지만, 생태계가 아직 동기 중심입니다.  
```

> 각 프레임워크는 설계 목적이 다릅니다.
> Flask는 가볍고 유연한 범용 웹 프레임워크, Django는 풀스택 웹 개발에 강합니다.
> FastAPI는 **API 서버 구축에 특화**되어 있으며,
> 모델 배포처럼 "입력 → 처리 → 응답" 구조의 서비스에 가장 잘 맞습니다.


### 1.4 FastAPI의 구성 요소


![image.png](images/nb/nb_04_171a20f0.png)

각 요소의 역할을 정리하겠습니다:

```
Uvicorn (ASGI 서버)
  - HTTP 요청을 받아서 FastAPI에 전달하는 역할입니다.
  - 웹 서버의 "문지기"라고 생각하면 됩니다.
  - FastAPI 자체는 요청을 받는 기능이 없으므로, Uvicorn이 필요합니다.
  * Uvicorn에 대한 자세한 설명은 추후에 다룹니다.

FastAPI (프레임워크)
  - URL 라우팅: 어떤 요청을 어떤 함수가 처리할지 결정합니다.
  - 데이터 검증: Pydantic을 사용하여 입력을 자동 검증합니다.
  - 문서 생성: Swagger UI를 자동으로 만들어줍니다.

엔드포인트 (여러분이 작성하는 코드)
  - 실제 비즈니스 로직이 담기는 함수입니다.
  - 모델 추론, 전처리, 후처리 등이 여기에 해당합니다.
  - 처음 만든 predict() 함수가 여기에 연결됩니다.
```

> 💡 **ASGI란?**
>
> ASGI (Asynchronous Server Gateway Interface)는 Python 비동기 웹 서버와
> 프레임워크 사이의 표준 인터페이스입니다.  
> Flask가 사용하는 WSGI(동기)의 비동기 버전이라고 이해하시면 됩니다.
> Uvicorn은 ASGI 서버의 대표적인 구현체입니다.


### 1.5 실습: 최소한의 FastAPI 서버 실행하기


이론은 충분합니다. 직접 서버를 띄워보겠습니다.


> ⚠️ **주피터 노트북에서의 FastAPI 실행**
>
> FastAPI 서버는 원래 터미널에서 `uvicorn app.main_basic:app`으로 실행합니다.
> 노트북 안에서 띄우려면 **백그라운드 스레드 + 자체 이벤트 루프**가 필요합니다.
> 이 노트북은 **섹션 0에서 정의한 `serve_in_thread()` 헬퍼**가 이 일을 대신해 줍니다
> (표준 asyncio 루프를 직접 써서 OS·`uvloop` 과 무관하게 안정적으로 동작, 커널을 죽이지 않음).
>
> 실제 프로젝트에서는 `.py` 파일에 작성하고 터미널에서 `uvicorn ... --reload`로 실행합니다.

#### 주피터 노트북에서 실행하는 방법 (이 노트북의 방식)

섹션 0에서 정의한 `serve_in_thread()`로 서버를 띄웁니다. 같은 포트에 서버가 떠 있으면 먼저 멈추고 새로 띄우므로, 셀을 다시 실행해도 됩니다.

```python
serve_in_thread("app.main_basic:app", port=8000)
```

참고: 원래 강의 자료는 `nest_asyncio`로 서버를 띄웠습니다. 그런데 macOS에서는 `uvicorn[standard]`가 함께 설치하는 `uvloop`과 `nest_asyncio`가 충돌해, 서버가 시작되지 않는 경우가 있습니다. 그래서 표준 asyncio 루프로 직접 띄우는 방식으로 바꿨습니다.

In [28]:
# 서버 실행 도우미 — 노트북 맨 처음에 한 번 실행하세요.
# 노트북 안에서 uvicorn 서버를 띄우고 멈추는 함수를 정의합니다.
import os, sys, asyncio, threading, time, socket, contextlib
import uvicorn

# 작업 디렉터리를 app/ 가 있는 위치로 맞춥니다 (notebooks/ 안에서 열어도 동작).
if not os.path.isdir('app') and os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
os.makedirs('app', exist_ok=True)

_SERVERS = {}  # port -> (server, thread)

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    """실행 중인 서버를 멈춥니다."""
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host='127.0.0.1', port=8000, log_level='warning'):
    """백그라운드에서 uvicorn 서버를 띄웁니다.

    app: FastAPI 객체 또는 'app.main:app' 같은 import 경로.
    같은 포트에 서버가 이미 있으면 먼저 멈추고 새로 띄웁니다.
    """
    stop_server(port)
    if isinstance(app, str):
        sys.modules.pop(app.split(':')[0], None)   # 파일을 다시 저장한 경우 최신 내용 반영
    for _ in range(50):
        if not _port_open(host, port):
            break
        time.sleep(0.1)
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop='asyncio')
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def _run():
        # Windows는 SelectorEventLoop, 그 외는 기본 이벤트 루프를 사용합니다.
        if sys.platform == 'win32':
            loop = asyncio.SelectorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    for _ in range(40):
        if _port_open(host, port):
            print(f'서버 실행됨: http://{host}:{port}')
            return server
        time.sleep(0.25)
    print('서버가 시작되지 않았습니다. 위 로그를 확인하세요.')
    return server

print('서버 도우미 준비 완료 (serve_in_thread, stop_server)')

서버 도우미 준비 완료 (serve_in_thread, stop_server)


In [ ]:
%%writefile app/main_basic.py
"""
최소한의 FastAPI 서버
"""
from fastapi import FastAPI

# FastAPI 인스턴스 생성
app = FastAPI(
    title="My First ML API",
    description="Day 2 실습: 첫 번째 FastAPI 서버",
    version="0.1.0",
)

# 엔드포인트 1: 헬스체크 (서버가 살아있는지 확인)
@app.get("/health")
def health_check():
    return {"status": "healthy"}

# 엔드포인트 2: 루트 경로
@app.get("/")
def root():
    return {
        "message": "ML Model Serving API",
        "docs_url": "/docs",
    }

In [31]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
serve_in_thread("app.main_basic:app", port=8000)

서버 실행됨: http://127.0.0.1:8000


![image.png](images/nb/nb_05_b82459cf.png)

In [30]:
# 참조를 잃어버린 uvicorn 서버를 gc 로 뒤져서 찾아내 멈춘다
import gc, time, uvicorn

servers = [o for o in gc.get_objects() if isinstance(o, uvicorn.Server)]
print(f"메모리에 남아 있는 uvicorn Server 객체: {len(servers)}개")

for s in servers:
    s.should_exit = True       # 이벤트 루프에게 "다음 틈에 종료하라"고 표시

# 포트가 실제로 풀릴 때까지 기다린다
for _ in range(50):
    if not _port_open('127.0.0.1', 8000):
        print("8000 풀렸다")
        break
    time.sleep(0.2)
else:
    print("아직 안 풀림 — 커널 재시작이 필요하다")

메모리에 남아 있는 uvicorn Server 객체: 5개
8000 풀렸다


#### 서버 코드 작성



#### 터미널에서 실행하는 방법 (표준 방식)

```bash
# 터미널에서 실행하는 표준 명령어입니다 (노트북이 아닌 별도 터미널에서)
# uvicorn app.main_basic:app --reload --port 8000

# --reload: 코드 변경 시 서버 자동 재시작 (개발 중에만 사용)
# --port 8000: 8000번 포트에서 서버 실행
```

![ce9a66c6-f26b-45dc-bf97-723f1df683fa.png](images/nb/nb_06_50ac4495.png)


#### 서버 호출 테스트



In [ ]:
import requests

# 헬스체크 엔드포인트 호출
response = requests.get("http://localhost:8000/health")
print(f"상태 코드: {response.status_code}")
print(f"응답: {response.json()}")


In [ ]:
# 루트 엔드포인트 호출
response = requests.get("http://localhost:8000/")
print(f"상태 코드: {response.status_code}")
print(f"응답: {response.json()}")


축하합니다. **첫 번째 API 서버가 동작하고 있습니다.**

이전 시간에 `requests.get()`으로 외부 API를 호출했던 것을 기억하시나요?  
이제 그 API가 **여러분의 컴퓨터에서 실행 중인 서버**입니다.


### 1.6 코드 해부: 한 줄씩 이해하기

방금 작성한 코드를 하나씩 뜯어보겠습니다.



```python
# 1. FastAPI 인스턴스 생성
app = FastAPI(
    title="My First ML API",           # Swagger UI에 표시될 제목
    description="Day 2 실습: ...",       # Swagger UI에 표시될 설명
    version="0.1.0",                    # API 버전
)
```


`app`은 FastAPI 애플리케이션 객체입니다.  
모든 엔드포인트는 이 `app`에 등록됩니다.  
`title`, `description`, `version`은 자동 생성되는 API 문서(Swagger UI)에 반영됩니다.


```python
# 2. 데코레이터로 엔드포인트 등록
@app.get("/health")       # GET 메서드, /health 경로
def health_check():       # 이 함수가 요청을 처리합니다
    return {"status": "healthy"}   # dict를 반환하면 자동으로 JSON 변환
```


`@app.get("/health")`는 두 가지를 선언합니다:
- **HTTP 메서드**: GET
- **URL 경로**: /health


이 선언에 의해, `GET /health` 요청이 들어오면 `health_check()` 함수가 실행됩니다.

함수의 반환값이 Python 딕셔너리이면, FastAPI가 자동으로 JSON으로 변환하여 응답합니다.  
앞서 배운 `json.dumps()`를 직접 호출할 필요가 없습니다.

```python
# 기존 방식: 수동으로 JSON 변환
import json
json_string = json.dumps({"status": "healthy"})

# FastAPI 방식: 그냥 dict를 return하면 됩니다
return {"status": "healthy"}    # FastAPI가 자동 변환
```


### ✅ 체크포인트

다음 질문에 답할 수 있다면, 이 섹션의 학습 목표를 달성한 것입니다:

1. FastAPI가 Flask보다 모델 배포에 적합한 이유 세 가지는 무엇입니까?
2. Uvicorn의 역할은 무엇이며, 왜 FastAPI와 함께 사용합니까?
3. `@app.get("/health")`에서 `get`과 `"/health"`는 각각 무엇을 의미합니까?
4. FastAPI에서 dict를 반환하면 어떤 일이 자동으로 일어납니까?

## 2. 첫 번째 엔드포인트 만들기: Path, Query, Body


> **학습 목표**
> - Path 파라미터, Query 파라미터, Request Body의 차이를 구분할 수 있습니다.
> - 각 파라미터 방식이 적합한 상황을 판단할 수 있습니다.
> - FastAPI에서 GET, POST 엔드포인트를 작성할 수 있습니다.
> - 모델 추론 API에 어떤 파라미터 방식을 사용해야 하는지 이해합니다.

### 2.1 세 가지 파라미터 방식 개요




클라이언트가 서버에 데이터를 전달하는 방식은 크게 세 가지입니다.

```
1. Path 파라미터    — URL 경로의 일부로 전달
   GET /models/sentiment-v1
                   ──────────── 이 부분

2. Query 파라미터   — URL 뒤에 ?key=value 형태로 전달
   GET /models?name=sentiment&version=1
               ─────────────────────── 이 부분

3. Request Body    — HTTP 요청 본문에 JSON으로 전달
   POST /predict
   {"text": "이 영화 재밌다"}   ← 이 부분
```


각각 언제 사용하는지 감을 잡기 위해, 모델 배포 상황에 대입해 보겠습니다.

```
"sentiment-v1 모델의 정보를 조회하고 싶다"
  → GET /models/sentiment-v1              (Path: 특정 리소스를 지정)

"모델 목록을 검색하되, 상태가 running인 것만 보고 싶다"
  → GET /models?status=running            (Query: 필터링 조건)

"이 텍스트의 감성을 분석해줘"
  → POST /predict  {"text": "..."}        (Body: 처리할 데이터)
```


이제 각각을 코드로 구현해 보겠습니다.



### 2.2 Path 파라미터 — URL 경로에 값을 포함

Path 파라미터는 URL 경로의 일부로 값을 전달하는 방식입니다.  
**특정 리소스를 식별**할 때 사용합니다.


#### 기본 문법

In [7]:
# app/ 폴더를 Python 패키지로 인식시키기 위해 __init__.py를 생성합니다.
# 이 파일이 없으면 uvicorn이 app.main_params를 import할 수 없습니다.
import os
os.makedirs("app", exist_ok=True)

with open("app/__init__.py", "w") as f:
    pass

print("✅ app/__init__.py 생성 완료")

✅ app/__init__.py 생성 완료


In [8]:
%%writefile app/main_params.py
"""
파라미터 방식 실습
"""
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional

app = FastAPI(title="Parameter Examples")

# ===== Path 파라미터 =====

# 기본 사용: 중괄호 {}로 경로 변수를 선언합니다
@app.get("/models/{model_name}")
def get_model_info(model_name: str):
    """특정 모델의 정보를 반환합니다."""
    return {
        "model_name": model_name,
        "status": "running",
        "version": "1.0.0",
    }

Overwriting app/main_params.py


In [9]:
# 이전에 띄운 서버가 있으면 멈춥니다 (serve_in_thread 가 자동으로도 처리합니다).
stop_server(8000)

In [10]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
serve_in_thread("app.main_params:app", port=8000)

서버 실행됨: http://127.0.0.1:8000


In [11]:
import requests

# Path 파라미터 테스트
response = requests.get("http://localhost:8000/models/sentiment-v1")
print(response.json())
# {'model_name': 'sentiment-v1', 'status': 'running', 'version': '1.0.0'}

response = requests.get("http://localhost:8000/models/image-classifier")
print(response.json())
# {'model_name': 'image-classifier', 'status': 'running', 'version': '1.0.0'}

{'model_name': 'sentiment-v1', 'status': 'running', 'version': '1.0.0'}
{'model_name': 'image-classifier', 'status': 'running', 'version': '1.0.0'}


작성자의 경우, 여러 포트를 바꿔 실험하다보니 충돌 상황을 겪었습니다만, 수행에는 지장이 없었습니다

URL에 포함된 `sentiment-v1`, `image-classifier`가 함수의 `model_name` 파라미터로 자동 전달됩니다.


#### 타입 지정의 효과



FastAPI는 타입 힌트를 실제 검증에 활용합니다.



```python
# app/main_params.py에 아래 엔드포인트를 추가합니다 (%%writefile -a 사용)
```

```python
%%writefile -a app/main_params.py

# Path 파라미터에 int 타입 지정
@app.get("/predictions/{prediction_id}")
def get_prediction(prediction_id: int):
    """특정 예측 결과를 조회합니다."""
    return {
        "prediction_id": prediction_id,
        "label": "긍정",
        "confidence": 0.92,
    }
```


In [12]:
%%writefile -a app/main_params.py

# Path 파라미터에 int 타입 지정
@app.get("/predictions/{prediction_id}")
def get_prediction(prediction_id: int):
    """특정 예측 결과를 조회합니다."""
    return {
        "prediction_id": prediction_id,
        "label": "긍정",
        "confidence": 0.92,
    }

Appending to app/main_params.py


In [13]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
serve_in_thread("app.main_params:app", port=8000)

서버 실행됨: http://127.0.0.1:8000


In [14]:
import requests

# 정상 요청: 숫자를 전달
response = requests.get("http://localhost:8000/predictions/42")
print(f"상태: {response.status_code}, 응답: {response.json()}")
# 상태: 200, 응답: {'prediction_id': 42, 'label': '긍정', 'confidence': 0.92}

# 잘못된 요청: 문자열을 전달
response = requests.get("http://localhost:8000/predictions/abc")
print(f"상태: {response.status_code}")
print(f"에러: {response.json()}")
# 상태: 422
# 에러: {'detail': [{'type': 'int_parsing', 'msg': 'Input should be a valid integer...'}]}

상태: 200, 응답: {'prediction_id': 42, 'label': '긍정', 'confidence': 0.92}
상태: 422
에러: {'detail': [{'type': 'int_parsing', 'loc': ['path', 'prediction_id'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'abc'}]}


`prediction_id: int`로 선언했기 때문에, 문자열 `"abc"`가 들어오면
FastAPI가 자동으로 422 에러를 반환합니다.  
별도의 검증 코드를 작성하지 않아도 됩니다.

#### Path 파라미터 사용 시점

```
적합한 경우:
  ✅ 특정 리소스를 식별할 때 — /models/{model_name}
  ✅ 계층 구조를 표현할 때 — /models/{model_name}/versions/{version_id}
  ✅ 값이 필수일 때 (URL 자체에 포함되므로 생략 불가)

부적합한 경우:
  ❌ 선택적인 필터링 조건 — Query 파라미터가 적합
  ❌ 복잡한 데이터 전달 — Request Body가 적합
```


### 2.3 Query 파라미터 — URL 뒤에 조건을 추가



Query 파라미터는 URL 뒤에 `?key=value` 형태로 전달하는 방식입니다.  
**검색, 필터링, 페이지네이션** 등에 사용합니다.



#### 기본 문법

In [15]:
%%writefile -a app/main_params.py

# ===== Query 파라미터 =====

# 함수 인자 중 Path에 포함되지 않은 것은 자동으로 Query 파라미터가 됩니다
@app.get("/models")
def list_models(status: str = None, limit: int = 10):
    """
    모델 목록을 조회합니다.

    - status: 필터링 조건 (선택) — "running", "stopped" 등
    - limit: 반환할 최대 개수 (기본값: 10)
    """
    # 실제로는 DB에서 조회하겠지만, 여기서는 예시 데이터를 반환합니다
    models = [
        {"name": "sentiment-v1", "status": "running"},
        {"name": "image-clf-v2", "status": "running"},
        {"name": "ner-v1", "status": "stopped"},
    ]

    # status 필터링
    if status:
        models = [m for m in models if m["status"] == status]

    # limit 적용
    models = models[:limit]

    return {
        "total": len(models),
        "models": models,
    }

Appending to app/main_params.py


In [16]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
serve_in_thread("app.main_params:app", port=8000)

서버 실행됨: http://127.0.0.1:8000


In [17]:
import requests

# 파라미터 없이 호출 (기본값 사용)
response = requests.get("http://localhost:8000/models")
print("전체 모델:", response.json())

# status로 필터링
response = requests.get("http://localhost:8000/models?status=running")
print("running만:", response.json())

# 여러 파라미터 조합
response = requests.get("http://localhost:8000/models?status=running&limit=1")
print("running, 1개만:", response.json())

전체 모델: {'total': 3, 'models': [{'name': 'sentiment-v1', 'status': 'running'}, {'name': 'image-clf-v2', 'status': 'running'}, {'name': 'ner-v1', 'status': 'stopped'}]}
running만: {'total': 2, 'models': [{'name': 'sentiment-v1', 'status': 'running'}, {'name': 'image-clf-v2', 'status': 'running'}]}
running, 1개만: {'total': 1, 'models': [{'name': 'sentiment-v1', 'status': 'running'}]}


핵심 포인트를 정리합니다:



```
- 기본값이 있는 파라미터(limit: int = 10) → 선택적 (생략 가능)
- 기본값이 None인 파라미터(status: str = None) → 선택적 (없으면 필터링 안 함)
- 기본값이 없는 파라미터(status: str) → 필수 (생략하면 422 에러)
```


#### Query 파라미터 사용 시점
```
적합한 경우:
  ✅ 검색 조건 — /models?name=sentiment
  ✅ 필터링 — /predictions?date=2024-01-01&label=긍정
  ✅ 페이지네이션 — /predictions?page=2&size=20
  ✅ 정렬 — /models?sort_by=accuracy&order=desc
  ✅ 선택적인 옵션 — /predict?return_probabilities=true

부적합한 경우:
  ❌ 리소스 식별 — Path 파라미터가 적합
  ❌ 대량의 데이터 전달 — URL 길이 제한이 있으므로 Request Body가 적합
```


### 2.4 Request Body — 본문에 JSON 데이터 전달

Request Body는 HTTP 요청의 본문에 JSON 데이터를 담아 보내는 방식입니다.  
**모델 추론 요청처럼 복잡한 데이터를 전달할 때** 사용합니다.


#### 기본 문법

FastAPI에서 Request Body를 받으려면 **Pydantic 모델**을 사용합니다.

In [18]:
%%writefile -a app/main_params.py

# ===== Request Body =====
from pydantic import BaseModel
from typing import Optional

# 입력 스키마 정의
class PredictRequest(BaseModel):
    text: str
    return_probabilities: bool = False    # 선택, 기본값 False

# 출력 스키마 정의
class PredictResponse(BaseModel):
    label: str
    confidence: float
    probabilities: Optional[dict] = None

@app.post("/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    """
    텍스트 감성 분석을 수행합니다.

    - text: 분석할 텍스트 (필수)
    - return_probabilities: 전체 확률을 반환할지 여부 (선택, 기본 False)
    """
    # 실제로는 모델 추론을 수행하겠지만, 여기서는 더미 결과를 반환합니다
    result = {
        "label": "긍정",
        "confidence": 0.92,
    }

    if request.return_probabilities:
        result["probabilities"] = {
            "긍정": 0.92,
            "부정": 0.05,
            "중립": 0.03,
        }

    return result

Appending to app/main_params.py


In [19]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
serve_in_thread("app.main_params:app", port=8000)

서버 실행됨: http://127.0.0.1:8000


In [20]:
import requests

# POST 요청: JSON 데이터를 본문에 담아 전송
response = requests.post(
    "http://localhost:8000/predict",
    json={"text": "이 영화 정말 재밌다"}
)
print("기본 응답:", response.json())

# 옵션 추가
response = requests.post(
    "http://localhost:8000/predict",
    json={
        "text": "이 영화 정말 재밌다",
        "return_probabilities": True
    }
)
print("확률 포함:", response.json())

기본 응답: {'label': '긍정', 'confidence': 0.92, 'probabilities': None}
확률 포함: {'label': '긍정', 'confidence': 0.92, 'probabilities': {'긍정': 0.92, '부정': 0.05, '중립': 0.03}}


#### 잘못된 요청 테스트


In [21]:

# text 필드 누락
response = requests.post(
    "http://localhost:8000/predict",
    json={"return_probabilities": True}
)
print(f"상태: {response.status_code}")
print(f"에러: {response.json()['detail'][0]['msg']}")
# 상태: 422
# 에러: Field required

# text에 잘못된 타입 전달
response = requests.post(
    "http://localhost:8000/predict",
    json={"text": 12345}
)
print(f"상태: {response.status_code}")
print(f"에러: {response.json()['detail'][0]['msg']}")
# 상태: 422
# 에러: Input should be a valid string

상태: 422
에러: Field required
상태: 422
에러: Input should be a valid string


> Pydantic이 자동으로 입력을 검증하고, 상세한 에러 메시지를 반환합니다.  
> 이 부분은 이후 섹션에서 더 깊이 다룹니다.



#### Request Body 사용 시점

```
적합한 경우:
  ✅ 모델 추론 요청 — 입력 데이터가 복잡하고 구조화되어 있을 때
  ✅ 여러 필드를 전달할 때 — 텍스트, 옵션, 설정 등
  ✅ 데이터 양이 많을 때 — URL 길이 제한이 없음
  ✅ 중첩된 데이터 구조 — JSON의 장점을 활용

부적합한 경우:
  ❌ 단순한 리소스 조회 — GET + Path/Query가 적합
  ❌ GET 요청 — 관례적으로 GET에는 Body를 사용하지 않음
```


### 2.5 정리: 언제 어떤 방식을 사용합니까?

![image.png](images/nb/nb_07_51abdc80.png)

 💡 **이 과정의 프로젝트에서 가장 많이 사용할 패턴**
>
> ```python
> @app.post("/predict")
> def predict(request: PredictRequest):    # Request Body (POST)
>     ...
> ```
>
> 모델 추론은 입력 데이터를 Body로 받는 POST 요청이 기본 패턴입니다.


### ✅ 체크포인트

다음 질문에 답할 수 있다면, 이 섹션의 학습 목표를 달성한 것입니다:

1. `/models/sentiment-v1`에서 `sentiment-v1`은 어떤 종류의 파라미터입니까?
2. `/models?status=running&limit=5`에서 `status`와 `limit`은 어떤 종류의 파라미터입니까?
3. 모델 추론 요청에 Request Body를 사용하는 이유는 무엇입니까?
4. FastAPI에서 함수의 파라미터가 Path, Query, Body 중 어디서 오는지 어떻게 판별합니까?


## 3. Swagger UI로 API 테스트하기



> **학습 목표**
> - Swagger UI가 무엇이고, FastAPI에서 어떻게 자동 생성되는지 이해합니다.
> - Swagger UI에서 GET, POST 엔드포인트를 직접 호출할 수 있습니다.
> - API 문서가 코드와 자동으로 동기화되는 원리를 이해합니다.
> - ReDoc과 OpenAPI 스펙의 존재를 알고, 활용 시점을 파악합니다.


### 3.1 왜 API 문서가 중요합니까?

API를 만들었으면, 누군가 그 API를 **사용**해야 합니다.
사용하려면 다음 정보가 필요합니다:



```
- 어떤 URL로 요청해야 하는가?
- 어떤 HTTP 메서드를 사용하는가?
- 어떤 데이터를 보내야 하는가? (입력 형식)
- 어떤 데이터가 돌아오는가? (출력 형식)
- 에러가 발생하면 어떤 응답이 오는가?
```


전통적으로는 이 정보를 **수동으로 문서화**했습니다.  
Word, Notion, Wiki 등에 API 스펙을 작성하고, 코드가 바뀔 때마다 문서도 업데이트해야 했습니다.  

문제는 명확합니다:  



```
코드가 바뀌었는데 문서를 업데이트하지 않으면?
  → 문서와 실제 동작이 다릅니다.
  → 클라이언트가 잘못된 형식으로 요청합니다.
  → "문서대로 했는데 안 돼요" 문의가 쏟아집니다.
```



FastAPI는 이 문제를 **코드에서 문서를 자동 생성**하는 방식으로 해결합니다.




### 3.2 Swagger UI 접속하기


섹션 2에서 만든 서버가 실행 중인 상태에서, 브라우저를 열고 다음 주소로 접속합니다.

```
http://localhost:8000/docs
```

> ⚠️ **Colab 환경에서의 접속**
>
> Google Colab에서는 `localhost`에 직접 접속할 수 없습니다.
> 대신 노트북 셀에서 아래 코드를 실행하여 공개 URL을 얻을 수 있습니다.

> **로컬 환경**: 브라우저에서 바로 `http://localhost:8000/docs` 에 접속하면 됩니다.
>
> (Google Colab을 쓰는 경우에만) 외부 접속 URL 생성:
> ```python
> from google.colab import output
> output.serve_kernel_port_as_iframe(8000, path='/docs')
> ```

> 또는 로컬 환경에서 실습하시는 분은 브라우저에서 바로 접속하시면 됩니다.


In [22]:
from google.colab import output
output.serve_kernel_port_as_iframe(8000, path='/docs')

ModuleNotFoundError: No module named 'google.colab'

In [23]:
# 코랩 전용 셀이라 로컬에서는 못 쓴다. 대신 서버가 실제로 응답하는지 직접 확인한다.
import requests

# Swagger UI 페이지가 뜨는지
r = requests.get("http://localhost:8000/docs")
print(f"/docs 상태 코드: {r.status_code}")

# Swagger UI 가 읽어들이는 원본 스펙에서 엔드포인트 목록 뽑기
spec = requests.get("http://localhost:8000/openapi.json").json()
print(f"\n문서 제목: {spec['info']['title']}")
print("등록된 엔드포인트:")
for path, ops in spec["paths"].items():
    for method in ops:
        print(f"  {method.upper():5} {path}")

# 브라우저로는 http://localhost:8000/docs 에 직접 접속해서 캡처한다

/docs 상태 코드: 200

문서 제목: Parameter Examples
등록된 엔드포인트:
  GET   /models/{model_name}
  GET   /predictions/{prediction_id}
  GET   /models
  POST  /predict


접속하면 다음과 같은 화면이 나타납니다:


![a9acf980-0f7d-4626-9353-38e88b7218ae.png](images/nb/nb_08_27b8570c.png)

코드 한 줄 추가 없이, 섹션 2에서 작성한 모든 엔드포인트가 문서화되어 있습니다.
함수의 docstring이 설명으로, Pydantic 모델이 스키마로, 타입 힌트가 파라미터 정보로 반영됩니다.


![3ae76603-1043-4f6f-80b2-11480fc27dbb.png](images/nb/nb_09_bb53f609.png)


### 3.3 실습: Swagger UI에서 API 호출하기

Swagger UI의 가장 강력한 기능은 **브라우저에서 바로 API를 테스트**할 수 있다는 것입니다.

#### GET 요청 테스트

```
Step 1: GET /models/{model_name} 항목을 클릭하여 펼칩니다.
Step 2: 우측 상단의 [Try it out] 버튼을 클릭합니다.
Step 3: model_name 입력란에 sentiment-v1 을 입력합니다.
Step 4: [Execute] 버튼을 클릭합니다.
```

실행 결과가 바로 아래에 표시됩니다:


![a7f1499c-1bac-411d-a5c1-99b856a28c0c.png](images/nb/nb_10_44e6b9a1.png)


```
Responses

  Code    Description
  200     Successful Response

  Response body:
  {
      "model_name": "sentiment-v1",
      "status": "running",
      "version": "1.0.0"
  }

  Response headers:
    content-type: application/json

  Curl:
    curl -X 'GET' 'http://localhost:8000/models/sentiment-v1' -H 'accept: application/json'
```


> 💡 **Curl 명령어 자동 생성**
>
> Swagger UI는 실행할 때마다 해당 요청의 `curl` 명령어를 함께 보여줍니다.
> 이 명령어를 복사하면 터미널에서 동일한 요청을 재현할 수 있습니다.
> 프론트엔드 개발자에게 "이렇게 호출하면 됩니다"라고 전달할 때 유용합니다.



#### POST 요청 테스트

```
Step 1: POST /predict 항목을 클릭하여 펼칩니다.
Step 2: [Try it out] 버튼을 클릭합니다.
Step 3: Request body 영역에 JSON을 입력합니다:
```


```json
{
    "text": "이 영화 정말 재밌다",
    "return_probabilities": true
}
```


```
Step 4: [Execute] 버튼을 클릭합니다.
```


```
Responses

  Code    Description
  200     Successful Response

  Response body:
  {
      "label": "긍정",
      "confidence": 0.92,
      "probabilities": {
          "긍정": 0.92,
          "부정": 0.05,
          "중립": 0.03
      }
  }
```

![01f9eaf3-ee5a-4111-aa77-c71aa86b8141.png](images/nb/nb_11_b1d587fd.png)


#### 에러 상황 테스트

```
Step 1: POST /predict에서 [Try it out]
Step 2: Request body에 잘못된 데이터를 입력합니다:
```

```json
{
    "return_probabilities": true
}
```


```
Step 3: [Execute]
```

```
Responses

  Code    Description
  422     Validation Error

  Response body:
  {
      "detail": [
          {
              "type": "missing",
              "loc": ["body", "text"],
              "msg": "Field required",
              "input": {"return_probabilities": true}
          }
      ]
  }
```

![7827fa65-391b-45b7-95a5-027c45ea29ca.png](images/nb/nb_12_6e6f68cb.png)


`text` 필드가 누락되었다는 에러가 상세하게 표시됩니다.  
어떤 필드가(`loc`), 왜(`msg`) 문제인지 명확히 알 수 있습니다.
:


### 3.4 문서가 코드에서 자동 생성되는 원리

Swagger UI가 "마법"처럼 느껴질 수 있지만, 원리는 단순합니다.

![image.png](images/nb/nb_13_2a235617.png)

실제로 확인해 보겠습니다.


In [24]:
# FastAPI가 자동 생성한 OpenAPI 스펙 확인
import requests, json

response = requests.get("http://localhost:8000/openapi.json")
spec = response.json()

print(f"API 제목: {spec['info']['title']}")
print(f"API 버전: {spec['info']['version']}")
print(f"\n등록된 엔드포인트:")
for path, methods in spec['paths'].items():
    for method in methods:
        print(f"  {method.upper():6s} {path}")

API 제목: Parameter Examples
API 버전: 0.1.0

등록된 엔드포인트:
  GET    /models/{model_name}
  GET    /predictions/{prediction_id}
  GET    /models
  POST   /predict


In [25]:
# PredictRequest의 JSON Schema 확인
predict_schema = spec['components']['schemas']['PredictRequest']
print("PredictRequest 스키마:")
print(json.dumps(predict_schema, indent=2, ensure_ascii=False))

PredictRequest 스키마:
{
  "properties": {
    "text": {
      "type": "string",
      "title": "Text"
    },
    "return_probabilities": {
      "type": "boolean",
      "title": "Return Probabilities",
      "default": false
    }
  },
  "type": "object",
  "required": [
    "text"
  ],
  "title": "PredictRequest"
}


> **핵심**: 아래처럼 Swagger UI는 이 JSON Schema를 읽어서 입력 폼을 자동으로 만듭니다.  
> 여러분이 Pydantic 모델을 수정하면, 문서도 자동으로 업데이트됩니다.  
> 코드와 문서가 항상 동기화됩니다.  

![cbac4d01-627d-45a8-8925-265605f341e8.png](images/nb/nb_14_c1f27cd9.png)


### 3.5 문서 품질 높이기: 알아두면 유용한 옵션들

자동 생성된 문서만으로도 충분하지만, 몇 가지 옵션을 추가하면 문서가 더 명확해집니다.  
여기서는 자주 쓰이는 것만 예시로 간략히 소개합니다.


```python
from pydantic import BaseModel, Field

# Field()에 description과 examples를 추가하면 Swagger UI에 반영됩니다
class PredictRequest(BaseModel):
    text: str = Field(
        ...,
        min_length=1,
        max_length=5000,
        description="분석할 텍스트. 1자 이상 5000자 이하.",
        examples=["이 영화 정말 재밌다"],
    )
    return_probabilities: bool = Field(
        default=False,
        description="True로 설정하면 각 클래스별 확률을 함께 반환합니다.",
    )
```

> ⚠️ **아래 코드는 실행하지 마세요.**  
> 실제 서버 코드(`main_params.py`)에서 `summary=`를 이렇게 추가할 수 있다는 예시입니다.  
> 노트북에서 직접 실행하면 `app`이 정의되어 있지 않아 에러가 발생합니다.  


```python
# 엔드포인트에 summary를 추가하면 Swagger UI에서 짧은 제목으로 표시됩니다
@app.post("/predict", summary="텍스트 감성 분석")
def predict(request: PredictRequest):
    """입력된 텍스트의 감성을 분석합니다."""
    ...
```


> 💡 **실무 팁**
>
> `Field(description=, examples=)`를 습관적으로 넣는 것만으로도
 동료가 API를 이해하는 데 걸리는 시간이 크게 줄어듭니다.  
> 더 고급 옵션(`tags`, `model_config`, `json_schema_extra` 등)은
 FastAPI 공식 문서에서 필요할 때 참고하시면 됩니다.  


### 3.6 ReDoc — 또 다른 자동 문서

FastAPI는 Swagger UI 외에 **ReDoc**이라는 문서도 자동으로 생성합니다.

```
Swagger UI: http://localhost:8000/docs
ReDoc:      http://localhost:8000/redoc
```

**로컬 환경**: 브라우저에서 `http://localhost:8000/redoc` 에 접속하세요.

```python
# (Google Colab에서만 필요) — 로컬에서는 위 주소로 바로 접속
# from google.colab import output
# output.serve_kernel_port_as_iframe(8000, path='/redoc')
```

> ReDoc 페이지가 빈 화면으로 나오는 경우:  
> ReDoc은 외부 CDN에서 JavaScript를 로드하는데, 네트워크 환경에 따라   
> 차단될 수 있습니다. 이 경우 /docs (Swagger UI)를 사용하시면 됩니다.  
> 이 과정에서는 Swagger UI만 사용하므로 진행에 지장은 없습니다.  
  
아래 코드 결과가 잘 출력된다면, redoc이 정상적으로 작동중인 것입니다.  

In [26]:
import requests
resp = requests.get("http://localhost:8000/redoc")
print(f"상태: {resp.status_code}")
print(f"내용 길이: {len(resp.text)}")

상태: 200
내용 길이: 902


두 문서의 차이를 정리합니다:


![image.png](images/nb/nb_15_aee5e338.png)


> 이 과정에서는 **Swagger UI(/docs)** 를 기본으로 사용합니다.  
> 직접 API를 호출해볼 수 있어 개발과 테스트에 훨씬 편리하기 때문입니다.


### ✅ 체크포인트

다음 질문에 답할 수 있다면, 이 섹션의 학습 목표를 달성한 것입니다:

1. FastAPI에서 Swagger UI에 접속하려면 어떤 URL로 이동합니까?
2. Swagger UI가 코드와 항상 동기화될 수 있는 이유는 무엇입니까?
3. Pydantic 모델의 `Field(description=, examples=)`는 Swagger UI의 어디에 반영됩니까?
4. Swagger UI와 ReDoc의 핵심 차이는 무엇입니까?

## 4. Pydantic을 활용한 입력 데이터 검증(Schema)



> **학습 목표**
> - Pydantic이 무엇이고, FastAPI에서 어떤 역할을 하는지 설명할 수 있습니다.
> - 다양한 검증 규칙(타입, 범위, 길이, 패턴 등)을 Pydantic 모델로 정의할 수 있습니다.
> - 중첩된 스키마와 선택적 필드를 설계할 수 있습니다.
> - 커스텀 Validator를 작성하여 복잡한 검증 로직을 구현할 수 있습니다.
> - 검증 실패 시 반환되는 에러 응답의 구조를 이해합니다.


### 4.1 왜 입력 검증이 중요합니까?

모델 추론 API에 이런 요청이 들어왔다고 가정해 보겠습니다.

```python
# 정상적인 요청
{"text": "이 영화 정말 재밌다"}

# 비정상적인 요청들
{"text": ""}                           # 빈 문자열
{"text": 12345}                        # 잘못된 타입
{}                                     # 필수 필드 누락
{"text": "a" * 1000000}               # 비정상적으로 긴 입력
{"text": "분석해줘", "lang": "zz"}      # 존재하지 않는 언어 코드
```

이런 요청이 검증 없이 모델까지 도달하면 어떻게 될까요?

```
빈 문자열         → 토크나이저가 빈 텐서를 생성 → 모델 에러 또는 무의미한 결과
잘못된 타입       → 전처리 코드에서 TypeError 발생 → 500 Internal Server Error
필수 필드 누락    → KeyError 발생 → 500 Internal Server Error
비정상적 긴 입력  → 메모리 부족 → 서버 다운
잘못된 옵션 값    → 예상치 못한 동작 → 디버깅 어려움
```


공통된 결과는 두 가지입니다:

```
1. 서버가 500 에러를 반환합니다.
   → 클라이언트는 "내가 뭘 잘못 보냈는지" 알 수 없습니다.

2. 최악의 경우 서버가 다운됩니다.
   → 다른 정상 사용자도 서비스를 이용할 수 없게 됩니다.
```


> **입력 검증의 목적** 은 잘못된 데이터가 모델에 도달하기 전에 차단하고,
> 클라이언트에게 **무엇이 잘못되었는지** 명확히 알려주는 것입니다.


![image.png](images/nb/nb_16_d7b06c0a.png)

### 4.2 Pydantic 기초: BaseModel



Pydantic은 Python의 타입 힌트를 기반으로 **데이터 검증과 직렬화**를 수행하는 라이브러리입니다.  
FastAPI에 내장되어 있으며, 별도 설치가 필요 없습니다.

#### 기본 사용법


In [32]:

from pydantic import BaseModel
from typing import Optional

# Pydantic 모델 정의 = 데이터의 "설계도"
class PredictRequest(BaseModel):
    text: str                              # 필수, 문자열
    language: str = "ko"                   # 선택, 기본값 "ko"
    return_probabilities: bool = False     # 선택, 기본값 False
    top_k: Optional[int] = None            # 선택, 없으면 None


이 네 필드가 보여주는 핵심 패턴 세 가지입니다:  
```
타입만 선언 (text: str)           → 필수 필드 (없으면 에러)
기본값 지정 (language: str = "ko") → 선택적 필드 (없으면 기본값 사용)
Optional + None                   → "없을 수도 있는" 선택적 필드
```

In [33]:
# 정상적인 데이터로 인스턴스 생성
req = PredictRequest(text="이 영화 재밌다")
print(f"text: {req.text}")
print(f"language: {req.language}")               # 기본값 "ko"
print(f"return_probabilities: {req.return_probabilities}")  # 기본값 False
print(f"top_k: {req.top_k}")                     # 기본값 None

text: 이 영화 재밌다
language: ko
return_probabilities: False
top_k: None


In [34]:
# 자동 타입 변환
req2 = PredictRequest(text="테스트", top_k="3")   # 문자열 "3"이 int 3으로 변환
print(f"top_k: {req2.top_k}, 타입: {type(req2.top_k)}")
# top_k: 3, 타입:

top_k: 3, 타입: <class 'int'>


In [36]:
# Pydantic 이 어느 방향으로는 변환해주고 어느 방향은 거절하는지 직접 확인한다
from pydantic import BaseModel, ValidationError

class Probe(BaseModel):
    s: str = ""
    i: int = 0
    b: bool = False

cases = [
    ("i", "123", "문자열 '123' 을 int 로"),
    ("s", 123,   "숫자 123 을 str 로"),
    ("b", "yes", "문자열 'yes' 를 bool 로"),
    ("b", "네",  "문자열 '네' 를 bool 로"),
    ("i", 3.0,   "실수 3.0 을 int 로"),
    ("i", 3.7,   "실수 3.7 을 int 로"),
]

for field, value, desc in cases:
    try:
        m = Probe(**{field: value})          # 그 필드 하나만 넣어서 만들어 본다
        print(f"통과   {desc:24s} -> {getattr(m, field)!r}")
    except ValidationError as e:
        print(f"거절   {desc:24s} -> {e.errors()[0]['type']}")   # 거절 사유 코드

통과   문자열 '123' 을 int 로        -> 123
거절   숫자 123 을 str 로           -> string_type
통과   문자열 'yes' 를 bool 로       -> True
거절   문자열 '네' 를 bool 로         -> bool_parsing
통과   실수 3.0 을 int 로           -> 3
거절   실수 3.7 을 int 로           -> int_from_float


In [35]:
# 검증 실패: 필수 필드 누락
from pydantic import ValidationError

try:
    req3 = PredictRequest()   # text가 없음
except ValidationError as e:
    print("검증 실패!")
    print(e)

검증 실패!
1 validation error for PredictRequest
text
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


> Pydantic의 핵심 동작을 정리하면:
> - 타입이 맞으면 → 통과
> - 변환 가능하면 → 자동 변환 후 통과 (예: `"3"` → `3`)
> - 변환도 불가능하면 → `ValidationError` 발생

### 4.3 Field를 활용한 상세 검증


`Field()`를 사용하면 타입 검사를 넘어 **값의 범위, 길이, 패턴** 등을 세밀하게 제어할 수 있습니다.


#### 문자열 검증


In [37]:
from pydantic import BaseModel, Field

class TextInput(BaseModel):
    text: str = Field(
        ...,                        # ... 은 "필수"를 의미합니다
        min_length=1,               # 최소 1글자 (빈 문자열 방지)
        max_length=5000,            # 최대 5000자 (비정상 입력 방지)
        description="분석할 텍스트",
    )


In [38]:
# 테스트
try:
    TextInput(text="")   # 빈 문자열
except ValidationError as e:
    print(f"빈 문자열: {e.errors()[0]['msg']}")
# 빈 문자열: String should have at least 1 character

try:
    TextInput(text="a" * 5001)   # 5000자 초과
except ValidationError as e:
    print(f"초과: {e.errors()[0]['msg']}")
# 초과: String should have at most 5000 characters


빈 문자열: String should have at least 1 character
초과: String should have at most 5000 characters


#### 숫자 검증


In [39]:
class InferenceOptions(BaseModel):
    temperature: float = Field(
        default=1.0,
        gt=0.0,       # greater than: 0보다 커야 함
        le=2.0,        # less than or equal: 2 이하
        description="생성 온도. 0 초과, 2 이하.",
    )
    top_k: int = Field(
        default=5,
        ge=1,          # greater than or equal: 1 이상
        le=100,        # less than or equal: 100 이하
        description="반환할 상위 결과 수",
    )
    batch_size: int = Field(
        default=1,
        ge=1,
        le=32,
        description="배치 크기. 1 이상 32 이하.",
    )

In [40]:
# 정상
opts = InferenceOptions(temperature=0.7, top_k=10)
print(f"temperature: {opts.temperature}, top_k: {opts.top_k}")

# 범위 초과
try:
    InferenceOptions(temperature=0.0)   # gt=0.0 이므로 0은 불가
except ValidationError as e:
    print(f"에러: {e.errors()[0]['msg']}")
# 에러: Input should be greater than 0

try:
    InferenceOptions(top_k=0)   # ge=1 이므로 0은 불가
except ValidationError as e:
    print(f"에러: {e.errors()[0]['msg']}")
# 에러: Input should be greater than or equal to 1

temperature: 0.7, top_k: 10
에러: Input should be greater than 0
에러: Input should be greater than or equal to 1


![image.png](images/nb/nb_17_90d4bcef.png)

#### 선택지 제한: Literal


특정 값만 허용해야 할 때는 `Literal`을 사용합니다.


In [41]:
from typing import Literal

class AnalysisRequest(BaseModel):
    text: str = Field(..., min_length=1)
    language: Literal["ko", "en", "ja"] = Field(
        default="ko",
        description="지원 언어: ko(한국어), en(영어), ja(일본어)",
    )
    task: Literal["sentiment", "summary", "ner"] = Field(
        ...,
        description="수행할 작업 유형",
    )



In [42]:
# 정상
req = AnalysisRequest(text="테스트", task="sentiment")
print(f"language: {req.language}, task: {req.task}")

# 허용되지 않은 값
try:
    AnalysisRequest(text="test", language="fr", task="sentiment")
except ValidationError as e:
    print(f"에러: {e.errors()[0]['msg']}")
# 에러: Input should be 'ko', 'en' or 'ja'

language: ko, task: sentiment
에러: Input should be 'ko', 'en' or 'ja'


### 4.4 응답 스키마와 response_model


입력뿐 아니라 출력도 Pydantic 모델로 정의할 수 있습니다.  
`response_model`로 지정하면 Swagger UI에 응답 형식이 자동 문서화됩니다.


```python
class PredictResponse(BaseModel):
    label: str = Field(description="예측 레이블")
    confidence: float = Field(description="확신도 (0.0 ~ 1.0)", ge=0.0, le=1.0)

> ⚠️ **아래 코드는 실행하지 마세요.**  
> 실제 서버 코드(`main_params.py`)에서 `summary=`를 이렇게 추가할 수 있다는 예시입니다.  
> 노트북에서 직접 실행하면 `app`이 정의되어 있지 않아 에러가 발생합니다.  


```python
@app.post("/predict", response_model=PredictResponse)
def predict(request: TextInput):
    # 추론 로직...
    return PredictResponse(label="긍정", confidence=0.92)
```


> `response_model`을 지정하면:
> - Swagger UI에 응답 스키마가 표시됩니다.
> - 스키마에 정의되지 않은 필드는 응답에서 자동으로 제거됩니다.
> - 내부 데이터가 실수로 클라이언트에 노출되는 것을 방지합니다.

---


### 4.5 422 에러 응답의 구조

FastAPI + Pydantic이 반환하는 422 에러의 구조를 이해해 두면
디버깅할 때 "왜 422가 뜨지?"에서 멈추지 않을 수 있습니다.

```python
# 의도적으로 잘못된 요청을 보내서 에러 구조를 확인합니다
import requests, json

response = requests.post(
    "http://localhost:8000/predict",
    json={
        "return_probabilities": "yes"   # bool이어야 하는데 문자열
        # text 필드 누락
    }
)

print(f"상태 코드: {response.status_code}")
print(f"에러 응답:")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

In [43]:
import requests, json

response = requests.post(
    "http://localhost:8000/predict",
    json={
        "return_probabilities": "yes"   # bool이어야 하는데 문자열
        # text 필드 누락
    }
)

print(f"상태 코드: {response.status_code}")
print(f"에러 응답:")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

상태 코드: 404
에러 응답:
{
  "detail": "Not Found"
}


에러 응답의 각 필드가 의미하는 바를 정리합니다:

```
detail          에러 목록 (여러 필드가 동시에 실패할 수 있으므로 배열)
  ├── type      에러 종류 ("missing", "string_type", "value_error" 등)
  ├── loc       에러 발생 위치 (["body", "text"] → 요청 본문의 text 필드)
  ├── msg       사람이 읽을 수 있는 에러 메시지
  └── input     클라이언트가 실제로 보낸 데이터
```

> 이 구조 덕분에 클라이언트는 **어떤 필드가, 왜 잘못되었는지** 정확히 파악할 수 있습니다.


### 4.6 정리: 이 과정에서 사용하는 Pydantic 패턴


![image.png](images/nb/nb_18_f7362118.png)

💡 **더 알고 싶다면**
>
> Pydantic은 이 외에도 다양한 기능을 제공합니다:
> - `Literal["ko", "en", "ja"]` — 허용된 값의 집합 제한
> - `@field_validator` — 단일 필드의 커스텀 검증 로직
> - `@model_validator` — 필드 간 교차 검증
> - 중첩 모델 — 복잡한 구조의 입력 데이터
>
> 이 기능들은 프로젝트가 복잡해질 때 필요에 따라 추가하면 됩니다.  
> Pydantic 공식 문서: https://docs.pydantic.dev


### ✅ 체크포인트

다음 질문에 답할 수 있다면, 이 섹션의 학습 목표를 달성한 것입니다:

1. `text: str`과 `text: str = "기본값"`의 차이는 무엇입니까?
2. `Field(..., min_length=1, max_length=5000)`에서 `...`은 무엇을 의미합니까?
3. 422 에러 응답에서 `loc` 필드는 어떤 정보를 담고 있습니까?
4. `response_model`을 지정하면 어떤 이점이 있습니까?

## 5. 실습: 모델 추론 엔드포인트 구현 및 테스트


실습 목표

- Day 1에서 저장한 모델과 추론 함수를 FastAPI 엔드포인트에 연결할 수 있습니다.  
- Pydantic 스키마로 입력/출력을 정의하고, 실제 모델 추론이 동작하는 API를 완성합니다.  
- Swagger UI와 Python 코드 두 가지 방법으로 추론 API를 테스트할 수 있습니다.  
- 에러 상황(잘못된 입력, 모델 로드 실패 등)에서 API가 어떻게 응답하는지 확인합니다.  

### 5.1 지금까지의 흐름 정리
오늘 배운 내용을 돌아보겠습니다.

- 섹션 1: FastAPI가 무엇이고 왜 쓰는지 배웠습니다.
- 섹션 2: Path, Query, Body 파라미터로 엔드포인트를 만들었습니다.
- 섹션 3: Swagger UI에서 API를 직접 호출해봤습니다.
- 섹션 4: Pydantic으로 입력 데이터를 검증하는 법을 익혔습니다.

지금까지는 더미 결과를 반환했습니다.  
`label: "긍정"`, `confidence: 0.92`를 하드코딩해서 돌려줬죠.  
이제 진짜 모델을 연결합니다.

> Day 1의 마지막에서 `app/model_utils.py`에 추론 함수를 분리해 두었습니다.  
> 오늘의 실습은 그 함수를 FastAPI 엔드포인트에서 호출하는 것입니다.



### 5.2 전체 구조 미리보기


![image.png](images/nb/nb_19_1a3f8286.png)

파일 구조는 다음과 같습니다:


model-serving-course/  
├── app/  
│   ├── main.py            # FastAPI 서버 (오늘 작성)  
│   └── model_utils.py     # 모델 로드 & 추론 (Day 1에서 작성)  
├── models/  
│   └── mnist_state_dict.pth   # Day 1에서 저장한 모델  
└── requirements.txt  

### 5.3 Step 1 — model_utils.py 확인 및 보완  
Day 1 실습에서 작성한 `model_utils.py`를 확인합니다.  
아직 작성하지 않았거나 내용이 다르다면, 아래 코드를 사용하세요.

> ⚠️ 이 파일은 Day 1의 마지막 섹션(5.6)에서 만든 것입니다.  
> Day 1을 건너뛰신 분은 아래 코드를 그대로 사용하시면 됩니다.

In [54]:
%%writefile app/model_utils.py
"""
모델 로드 및 추론 유틸리티
FastAPI 엔드포인트가 이 모듈을 import하여 사용합니다.
"""

import torch
import torch.nn as nn
from torchvision import transforms


# ===== 모델 정의 =====
class SimpleClassifier(nn.Module):
    """
    간단한 이미지 분류 모델
    - 입력: 1x28x28 (MNIST와 동일한 크기)
    - 출력: 10개 클래스에 대한 확률
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# ===== 전처리 파이프라인 =====
preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])


# ===== 모델 로드 =====
def load_model(model_path: str = "models/mnist_state_dict.pth") -> nn.Module:
    """
    저장된 state_dict를 로드하여 추론 가능한 모델을 반환합니다.
    """
    model = SimpleClassifier(num_classes=10)
    model.load_state_dict(
        torch.load(model_path, map_location="cpu", weights_only=True)
    )
    model.eval()
    return model

# ===== 전처리 상수 =====
# 학습할 때 쓴 값과 반드시 같아야 한다.
# 다르면 에러가 안 나고 예측 품질만 조용히 나빠진다.
MNIST_MEAN = 0.1307
MNIST_STD = 0.3081


# ===== [수정] 픽셀 리스트를 모델 입력 텐서로 =====
def to_model_input(pixel_values: list[float]) -> torch.Tensor:
    """
    0.0~1.0 범위의 픽셀 784개를 받아 모델이 먹는 텐서로 만든다.

    정규화를 클라이언트가 아니라 여기서 하는 이유:
    클라이언트에 맡기면 클라이언트마다 다르게 할 수 있는데,
    틀려도 200 OK 가 나가버려서 아무도 못 잡는다.
    """
    t = torch.tensor(pixel_values, dtype=torch.float32)
    t = t.reshape(1, 1, 28, 28)             # (배치, 채널, 높이, 너비)
    return (t - MNIST_MEAN) / MNIST_STD     # 학습 때와 똑같은 정규화


# ===== 추론 함수 =====
def predict(model: nn.Module, input_tensor: torch.Tensor) -> dict:
    """
    모델에 입력 텐서를 전달하고 예측 결과를 반환합니다.

    Args:
        model: 로드된 PyTorch 모델
        input_tensor: 전처리된 입력 텐서 (1, 1, 28, 28)

    Returns:
        dict: {"label": int, "confidence": float, "probabilities": list}
    """
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, dim=1)

    return {
        "label": predicted.item(),
        "confidence": round(confidence.item(), 4),
        "probabilities": probabilities[0].tolist(),
    }

Overwriting app/model_utils.py


작성 후 import가 정상적으로 되는지 확인합니다:


In [55]:
from app.model_utils import load_model, predict, preprocess
print("✅ model_utils import 성공")

# 모델 로드 테스트
model = load_model("models/mnist_state_dict.pth")
print(f"✅ 모델 로드 성공: {type(model).__name__}")

✅ model_utils import 성공
✅ 모델 로드 성공: SimpleClassifier


### 5.4 Step 2 — Pydantic 스키마 설계


API의 입력과 출력을 정의합니다. 섹션 4에서 배운 Pydantic을 실전에 적용하는 단계입니다.  
어떤 입력을 받아야 할까요?  
MNIST 모델은 28x28 흑백 이미지를 받습니다. 하지만 API 사용자가 PyTorch 텐서를 직접 보내지는 않습니다.  
실무에서는 이미지를 파일로 업로드하지만, 오늘은 기초 과정이므로 픽셀 값을 리스트로 전달하는 방식을 사용합니다.  

> 💡 이미지 파일 업로드는 추후에 다룹니다.  
> 오늘은 "모델과 API의 연결"에 집중합니다.  

In [56]:
%%writefile app/schemas.py
"""
API 입출력 스키마 정의
"""

from pydantic import BaseModel, Field
# from typing import Optional
from typing import Annotated, Optional


class PredictRequest(BaseModel):
    """모델 추론 요청 스키마"""
    # Annotated 로 리스트 "원소마다" 조건을 건다.
    # ge=0.0, le=1.0 -> 0 미만이거나 1 초과인 값이 하나라도 있으면 422.
    # 원본은 개수(784)만 검사해서, 정규화된 값을 보내도 그냥 통과했다.
    #pixel_values: list[float] = Field(
    pixel_values: list[Annotated[float, Field(ge=0.0, le=1.0)]] = Field(
        ...,
        min_length=784,       # 28 * 28 = 784
        max_length=784,
        #description="28x28 이미지의 픽셀 값 (784개). 0.0~1.0 범위.",
        description=(
              "28x28 이미지의 픽셀 값 784개. 0.0(검정)~1.0(흰색) 범위. "
              "정규화는 서버가 하므로 정규화하지 않은 값을 그대로 보낸다."
        ),
        examples=[[0.0] * 784],   # Swagger UI에 예시로 표시
    )
    return_probabilities: bool = Field(
        default=False,
        description="True로 설정하면 전체 클래스별 확률을 함께 반환합니다.",
    )


class PredictResponse(BaseModel):
    """모델 추론 응답 스키마"""
    label: int = Field(
        description="예측된 숫자 (0~9)",
    )
    confidence: float = Field(
        description="예측 확신도 (0.0~1.0)",
    )
    probabilities: Optional[list[float]] = Field(
        default=None,
        description="각 클래스(0~9)별 확률. return_probabilities=True일 때만 포함.",
    )
    model_version: str = Field(
        default="1.0.0",
        description="사용된 모델 버전",
    )


class HealthResponse(BaseModel):
    """헬스체크 응답 스키마"""
    status: str
    model_loaded: bool



Overwriting app/schemas.py


핵심 설계 결정을 짚고 넘어가겠습니다:  

- `pixel_values`는 정확히 784개여야 합니다. `min_length`와 `max_length`를 동일하게 설정하면 Pydantic이 자동으로 길이를 검증합니다.  
- `return_probabilities`는 선택적 옵션입니다. 기본값이 `False`이므로, 보내지 않아도 됩니다.  
- `PredictResponse`의 probabilities는 Optional입니다. 요청에 따라 포함 여부가 달라지기 때문입니다.

### 5.5 Step 3 — FastAPI 서버 작성

이제 본격적으로 서버 코드를 작성합니다. 지금까지 배운 모든 것이 들어갑니다.


In [57]:
%%writefile app/main.py
"""
Day 2 실습: 모델 추론 API 서버
"""

from fastapi import FastAPI, HTTPException
import torch

#from app.model_utils import load_model, predict
from app.model_utils import load_model, predict, to_model_input   # to_model_input 추가
from app.schemas import (
    PredictRequest,
    PredictResponse,
    HealthResponse,
)

# ===== FastAPI 앱 생성 =====
app = FastAPI(
    title="MNIST Prediction API",
    description="Day 2 실습: MNIST 숫자 분류 모델 추론 API",
    version="1.0.0",
)


# ===== 모델을 서버 시작 시 한 번만 로드 =====
# 모듈 레벨에서 로드하면 서버가 시작될 때 실행됩니다.
# 요청마다 로드하면 매번 수 초가 걸리므로, 반드시 한 번만 로드해야 합니다.
try:
    model = load_model("models/mnist_state_dict.pth")
    model_loaded = True
    print("✅ 모델 로드 완료")
except Exception as e:
    model = None
    model_loaded = False
    print(f"❌ 모델 로드 실패: {e}")


# ===== 엔드포인트 1: 헬스체크 =====
@app.get("/health", response_model=HealthResponse)
def health_check():
    """서버 상태와 모델 로드 여부를 확인합니다."""
    return HealthResponse(
        status="healthy",
        model_loaded=model_loaded,
    )


# ===== 엔드포인트 2: 모델 추론 =====
@app.post("/predict", response_model=PredictResponse, summary="MNIST 숫자 예측")
def predict_digit(request: PredictRequest):
    """
    28x28 이미지의 픽셀 값을 받아 숫자(0~9)를 예측합니다.

    - **pixel_values**: 784개의 float 리스트 (28x28 이미지)
    - **return_probabilities**: True로 설정하면 전체 확률 분포를 반환
    """
    # 1. 모델이 로드되었는지 확인
    if not model_loaded:
        raise HTTPException(
            status_code=503,
            detail="모델이 로드되지 않았습니다. 서버 로그를 확인하세요."
        )

    # 2. 입력 데이터를 텐서로 변환
    try:
        #input_tensor = torch.tensor(request.pixel_values, dtype=torch.float32)
        #input_tensor = input_tensor.reshape(1, 1, 28, 28)  # (batch, channel, H, W)
        input_tensor = to_model_input(request.pixel_values)
    except Exception as e:
        raise HTTPException(
            status_code=400,
            detail=f"입력 데이터를 텐서로 변환할 수 없습니다: {str(e)}"
        )

    # 3. 추론 실행
    try:
        result = predict(model, input_tensor)
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"모델 추론 중 에러가 발생했습니다: {str(e)}"
        )

    # 4. 응답 생성
    response = PredictResponse(
        label=result["label"],
        confidence=result["confidence"],
        model_version="1.0.0",
    )

    # 5. 옵션: 확률 분포 포함
    if request.return_probabilities:
        response.probabilities = [round(p, 4) for p in result["probabilities"]]

    return response

Overwriting app/main.py


**코드 해부: 핵심 포인트**  
**모델을 서버 시작 시 한 번만 로드하는 이유:**

```python
# 이렇게 하면 안 됩니다 (매 요청마다 모델 로드)
@app.post("/predict")
def predict_digit(request: PredictRequest):
    model = load_model(...)  # ❌ 요청마다 수 초 소요
    ...

# 이렇게 해야 합니다 (서버 시작 시 한 번만)
model = load_model(...)  # ✅ 서버 시작 시 1회 로드
@app.post("/predict")
def predict_digit(request: PredictRequest):
    result = predict(model, ...)  # 이미 로드된 모델 사용
```

모델 로드는 파일 I/O가 포함된 무거운 작업입니다. 요청마다 실행하면 응답 시간이 수 초로 늘어납니다.  
**HTTPException으로 에러를 명확히 전달하는 이유:**  

```
# 모델 미로드 → 503 (Service Unavailable)
raise HTTPException(status_code=503, detail="모델이 로드되지 않았습니다.")

# 입력 변환 실패 → 400 (Bad Request)
raise HTTPException(status_code=400, detail="입력 데이터를 변환할 수 없습니다.")

# 추론 실패 → 500 (Internal Server Error)
raise HTTPException(status_code=500, detail="모델 추론 중 에러가 발생했습니다.")
```


섹션 3에서 배운 HTTP 상태 코드를 기억하시나요?   
상황에 맞는 코드를 반환해야 클라이언트가 "내가 잘못한 건지, 서버가 문제인지" 구분할 수 있습니다.


### 5.6 Step 4 — 서버 실행 및 테스트


**서버 실행**

In [ ]:
# 서버 실행 (같은 포트에 서버가 떠 있으면 자동으로 멈추고 새로 띄웁니다)
#serve_in_thread("app.main:app", port=8000)

✅ 모델 로드 완료
서버 실행됨: http://127.0.0.1:8000


In [58]:
# app 패키지 밑 모듈들이 sys.modules 에 캐시돼 있으면 방금 고친 파일이 반영되지 않는다.
# 헬퍼는 'app.main' 하나만 지우기 때문에, model_utils 와 schemas 는 옛것이 그대로 남는다.
import sys

지울것 = [m for m in list(sys.modules) if m == "app" or m.startswith("app.")]
for m in 지울것:
    del sys.modules[m]
print(f"캐시에서 지운 모듈: {지울것}")

serve_in_thread("app.main:app", port=8000)


# 포트 말고 내용으로 확인한다

import requests
info = requests.get("http://localhost:8000/openapi.json").json()
print(info["info"]["title"])       # MNIST Prediction API 여야 한다

# 고친 범위 제약이 문서에 실제로 반영됐는지 본다
schema = info["components"]["schemas"]["PredictRequest"]["properties"]["pixel_values"]
print(schema)                       # items 에 minimum 0.0 / maximum 1.0 이 보여야 한다

캐시에서 지운 모듈: ['app', 'app.model_utils', 'app.schemas', 'app.main']
✅ 모델 로드 완료
서버 실행됨: http://127.0.0.1:8000
MNIST Prediction API
{'items': {'type': 'number', 'maximum': 1.0, 'minimum': 0.0}, 'type': 'array', 'maxItems': 784, 'minItems': 784, 'title': 'Pixel Values', 'description': '28x28 이미지의 픽셀 값 784개. 0.0(검정)~1.0(흰색) 범위. 정규화는 서버가 하므로 정규화하지 않은 값을 그대로 보낸다.', 'examples': [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0

```
터미널에서 실행하는 경우:
uvicorn app.main:app --reload --port 8000
```


**테스트 1: 헬스체크**


In [59]:
import requests

response = requests.get("http://localhost:8000/health")
print(f"상태 코드: {response.status_code}")
print(f"응답: {response.json()}")

상태 코드: 200
응답: {'status': 'healthy', 'model_loaded': True}


**테스트 2: 실제 MNIST 이미지로 추론**  
더미 데이터가 아닌, 실제 MNIST 테스트 이미지를 사용합니다.



In [60]:
from torchvision import datasets, transforms

# MNIST 테스트 데이터 로드
# [수정] Normalize 를 뺐다. 정규화는 이제 서버가 한다.
test_dataset = datasets.MNIST(
    root="data", train=False, download=True,
    transform=transforms.ToTensor()      # 0.0~1.0 로만 바꾼다
    #transform=transforms.Compose([
    #    transforms.ToTensor(),
    #    transforms.Normalize((0.1307,), (0.3081,)),
    #])
)

# 첫 번째 테스트 이미지 가져오기
test_image, true_label = test_dataset[0]
print(f"이미지 크기: {test_image.shape}")     # torch.Size([1, 28, 28])
print(f"정답 레이블: {true_label}")

# 픽셀 값을 리스트로 변환 (API에 보낼 형식)
pixel_values = test_image.flatten().tolist()
print(f"픽셀 값 개수: {len(pixel_values)}")   # 784


# 문서에 적힌 '0.0~1.0' 이 실제로 맞는지 직접 확인한다 (원본은 여기가 어긋나 있었다)
print(f"실제 픽셀 값 범위: {test_image.min():.4f} ~ {test_image.max():.4f}")

이미지 크기: torch.Size([1, 28, 28])
정답 레이블: 7
픽셀 값 개수: 784
실제 픽셀 값 범위: 0.0000 ~ 1.0000


이제 API를 호출합니다:


In [61]:
import json

# 추론 요청
response = requests.post(
    "http://localhost:8000/predict",
    json={
        "pixel_values": pixel_values,
        "return_probabilities": False,
    }
)

print(f"상태 코드: {response.status_code}")
print(f"응답:")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))



상태 코드: 200
응답:
{
  "label": 7,
  "confidence": 1.0,
  "probabilities": null,
  "model_version": "1.0.0"
}


 모델이 정확하게 예측했습니다.

**테스트 3: 확률 분포 포함 요청**


In [62]:
# return_probabilities를 True로 설정
response = requests.post(
    "http://localhost:8000/predict",
    json={
        "pixel_values": pixel_values,
        "return_probabilities": True,
    }
)

result = response.json()
print(f"예측: {result['label']} (확신도: {result['confidence']})")
print(f"\n클래스별 확률:")
for i, prob in enumerate(result['probabilities']):
    bar = "█" * int(prob * 50)
    print(f"  {i}: {prob:.4f} {bar}")

예측: 7 (확신도: 1.0)

클래스별 확률:
  0: 0.0000 
  1: 0.0000 
  2: 0.0000 
  3: 0.0000 
  4: 0.0000 
  5: 0.0000 
  6: 0.0000 
  7: 1.0000 ██████████████████████████████████████████████████
  8: 0.0000 
  9: 0.0000 


`return_probabilities` 옵션에 따라 응답 형식이 달라지는 것을 확인할 수 있습니다.
  

**테스트 4: 여러 이미지 연속 테스트**


In [63]:
# 10개 이미지를 연속으로 테스트
print(f"{'이미지':<8} {'정답':<6} {'예측':<6} {'확신도':<10} {'결과'}")
print("-" * 45)

correct = 0
for i in range(10):
    image, true_label = test_dataset[i]
    pixel_values = image.flatten().tolist()

    response = requests.post(
        "http://localhost:8000/predict",
        json={"pixel_values": pixel_values}
    )
    result = response.json()

    is_correct = result["label"] == true_label
    if is_correct:
        correct += 1

    mark = "✅" if is_correct else "❌"
    print(f"  #{i:<5} {true_label:<6} {result['label']:<6} {result['confidence']:<10} {mark}")

print(f"\n정확도: {correct}/10 ({correct * 10}%)")

이미지      정답     예측     확신도        결과
---------------------------------------------
  #0     7      7      1.0        ✅
  #1     2      2      1.0        ✅
  #2     1      1      1.0        ✅
  #3     0      0      0.9999     ✅
  #4     4      4      0.9996     ✅
  #5     1      1      1.0        ✅
  #6     4      4      0.9996     ✅
  #7     9      9      0.9998     ✅
  #8     5      5      0.9987     ✅
  #9     9      9      1.0        ✅

정확도: 10/10 (100%)


API를 통해 모델이 정상적으로 추론하는 것이 검증되었습니다.


### 5.7 Step 5 — 에러 상황 테스트
실서비스에서는 항상 잘못된 요청이 들어옵니다. API가 이를 어떻게 처리하는지 확인합니다.  
  
**에러 1: 픽셀 값 개수가 틀린 경우**

In [ ]:
# 784개가 아닌 100개만 전송
response = requests.post(
    "http://localhost:8000/predict",
    json={"pixel_values": [0.0] * 100}
)
print(f"상태 코드: {response.status_code}")  # 422
print(f"에러 메시지: {response.json()['detail'][0]['msg']}")

Pydantic이 `min_length=784 `조건에 의해 자동으로 거부했습니다.  
우리가 검증 코드를 한 줄도 작성하지 않았는데도 동작합니다.

**에러 2: 잘못된 데이터 타입**

In [ ]:
# 숫자가 아닌 문자열 전달
response = requests.post(
    "http://localhost:8000/predict",
    json={"pixel_values": "이것은 이미지가 아닙니다"}
)
print(f"상태 코드: {response.status_code}")  # 422

**에러 3: 필수 필드 누락**

In [ ]:
# pixel_values 없이 요청
response = requests.post(
    "http://localhost:8000/predict",
    json={"return_probabilities": True}
)
print(f"상태 코드: {response.status_code}")  # 422
print(f"에러: {response.json()['detail'][0]['msg']}")

**에러 4: 빈 JSON 전송**

In [ ]:
response = requests.post(
    "http://localhost:8000/predict",
    json={}
)
print(f"상태 코드: {response.status_code}")  # 422

모든 에러 상황에서 서버가 죽지 않고, 적절한 상태 코드와 메시지를 반환합니다.  
이것이 Pydantic + FastAPI 조합의 힘입니다.

### 5.8 Swagger UI에서 직접 테스트
브라우저에서도 동일한 테스트를 해봅니다.


브라우저에서 접속:
```
http://localhost:8000/docs
```
> (Google Colab에서만) 아래로 iframe 접속:
> ```python
> from google.colab import output
> output.serve_kernel_port_as_iframe(8000, path='/docs')
> ```

**Swagger UI 테스트 순서**    

1. POST /predict 항목을 클릭하여 펼칩니다.
2. [Try it out] 버튼을 클릭합니다.
3. Request body에 아래 JSON을 입력합니다:

![ac53b818-1764-4811-9d97-900d516620ab.png](images/nb/nb_20_187a7d07.png)

다음은 `/predict` 요청 본문의 예시입니다 (실행하는 코드가 아닙니다).

```json
{
    "pixel_values": [0.0, 0.0, 0.0, ... (784개)],
    "return_probabilities": true
}
```


💡 784개를 직접 입력하기 어렵다면, 위 테스트 코드에서 pixel_values를 출력하여 복사-붙여넣기 하세요:  

```
import json  
print(json.dumps(pixel_values))
```

4. [Execute] 버튼을 클릭합니다.  
5. 응답에서 예측 결과를 확인합니다.  

Swagger UI에서 입력 스키마가 자동으로 표시되는 것을 확인하세요.    
PredictRequest의 Field에 설정한 description, examples가 모두 반영되어 있습니다.  

### 5.9 정리: 오늘 완성한 것  
오늘 우리가 구현한 것을 돌아보겠습니다:


Day 1에서 준비한 것:  
  ✅ 학습된 MNIST 모델 (models/mnist_state_dict.pth)  
  ✅ 추론 함수 분리 (app/model_utils.py)  
  
Day 2에서 완성한 것:  
  ✅ Pydantic 스키마로 입출력 정의 (app/schemas.py)  
  ✅ FastAPI 서버에 추론 엔드포인트 연결 (app/main.py)  
  ✅ Swagger UI로 자동 문서 생성  
  ✅ 에러 처리: 잘못된 입력 → 422, 서버 문제 → 500/503  
  ✅ 실제 MNIST 이미지로 추론 검증  

**가장 중요한 변화:**  
주피터 노트북에서만 돌아가던 모델이, 이제 HTTP 요청으로 호출할 수 있는 API가 되었습니다.  
이것이 "모델 배포"의 첫 번째 이정표입니다.

✅ 체크포인트  
다음 질문에 답할 수 있다면, Day 2의 학습 목표를 달성한 것입니다:  

1. 모델을 서버 시작 시 한 번만 로드해야 하는 이유는 무엇입니까?  
2. pixel_values가 784개가 아닌 요청이 들어오면 어떤 일이 발생합니까? 이를 처리하는 코드를 직접 작성했습니까?  
3. HTTPException(status_code=503)은 어떤 상황에서 사용했습니까? 왜 500이 아니라 503입니까?  
4. Swagger UI에서 PredictRequest의 description과 examples가 어디에 표시됩니까  

### 제출

다음 내역을 MD 파일로 기록, 깃헙에 업로드하여 링크로 제출하시기 바랍니다  

1. 섹션 1.5 수행내역 캡쳐  
2. 섹션 2, 3 셀 출력  
3. 섹션 5 수행내역 캡쳐  
4. 각 섹션 체크포인트의 답변

수고하셨습니다!

In [ ]:
pixel_values